In [2]:
from local_utils import load_m49
import pandas as pd
import json

In [3]:
m49 = load_m49()

In [4]:
m49.head(2)

,region_id,region_parent,region_code,region_name
0,001,000,XA,World
1,002,001,XB,Africa


In [5]:
regions = m49[(m49['region_parent']=='001') & (m49['region_id']!='990')]
regions

,region_id,region_parent,region_code,region_name
1,002,001,XB,Africa
65,019,001,XC,Americas
126,142,001,XD,Asia
183,150,001,XE,Europe
243,009,001,XF,Oceania


In [6]:
subreg = m49[m49['region_parent'].isin(regions['region_id'])]
subreg

,region_id,region_parent,region_code,region_name
2,014,002,XH,Eastern Africa
22,017,002,XI,Middle Africa
32,015,002,XJ,Northern Africa
41,018,002,XK,Southern Africa
47,011,002,XL,Western Africa
66,029,019,XN,Caribbean
96,013,019,XO,Central America
105,005,019,XP,South America
120,021,019,XQ,Northern America
127,143,142,XR,Central Asia


In [7]:
countries = m49[m49['region_parent'].isin(subreg['region_id'])][['region_code', 'region_parent']].merge(subreg[['region_id', 'region_name']], left_on='region_parent', right_on='region_id').drop(columns=['region_parent', 'region_id']).set_index('region_code')
countries

,region_name
region_code,
BI,Eastern Africa
KM,Eastern Africa
DJ,Eastern Africa
ER,Eastern Africa
ET,Eastern Africa
...,...
WS,Polynesia
TK,Polynesia
TO,Polynesia


In [8]:
countries.to_parquet('m49-country-subregion.parquet')

In [9]:
with open('m49-country-subregion.json', 'w') as fp:
    json.dump(countries.to_dict()['region_name'], fp)

In [12]:
m49[m49['region_parent'].isin(subreg['region_id'])][['region_code', 'region_parent']].merge(subreg[['region_id', 'region_parent', 'region_name']], left_on='region_parent', right_on='region_id').merge(regions, left_on='region_parent_y', right_on='region_id')

,region_code_x,region_parent_x,region_id_x,region_parent_y,region_name_x,region_id_y,region_parent,region_code_y,region_name_y
0,BI,014,014,002,Eastern Africa,002,001,XB,Africa
1,KM,014,014,002,Eastern Africa,002,001,XB,Africa
2,DJ,014,014,002,Eastern Africa,002,001,XB,Africa
3,ER,014,014,002,Eastern Africa,002,001,XB,Africa
4,ET,014,014,002,Eastern Africa,002,001,XB,Africa
...,...,...,...,...,...,...,...,...,...
240,WS,061,061,009,Polynesia,009,001,XF,Oceania
241,TK,061,061,009,Polynesia,009,001,XF,Oceania
242,TO,061,061,009,Polynesia,009,001,XF,Oceania
243,TV,061,061,009,Polynesia,009,001,XF,Oceania


In [25]:
m49[m49['region_parent'].isin(subreg['region_id'])][['region_code', 'region_parent']]

,region_code,region_parent
3,BI,014
4,KM,014
5,DJ,014
6,ER,014
7,ET,014
...,...,...
268,WS,061
269,TK,061
270,TO,061
271,TV,061


In [45]:
subregion2continent = (subreg[['region_id', 'region_parent']]
    .merge(regions[['region_id', 'region_name']], left_on='region_parent', right_on='region_id')
    .drop(columns=['region_parent', 'region_id_y'])
    .rename(columns={'region_name':'continent_name', 'region_id_x': 'subregion_id'})
 )

In [48]:
cc2cont = (m49[m49['region_parent'].isin(subreg['region_id'])]
.loc[:,['region_code', 'region_parent']]
    .merge(subregion2continent, left_on='region_parent', right_on='subregion_id')
 .drop(columns=['region_parent', 'subregion_id'])
 .set_index('region_code')
)

In [50]:
with open('m49-country-continent.json', 'w') as fp:
    json.dump(cc2cont.to_dict()['continent_name'], fp)